In [1]:
from pyomo.environ import (Constraint,
                           Var,
                           ConcreteModel,
                           Expression,
                           Objective,
                           TransformationFactory,
                           value)
from pyomo.network import Arc, SequentialDecomposition
from idaes.core import FlowsheetBlock
from idaes.models.unit_models import (PressureChanger,
                                      Mixer,
                                      Separator as Splitter,
                                      Heater,
                                      StoichiometricReactor)
from idaes.models.unit_models import Flash
from idaes.models.unit_models.pressure_changer import ThermodynamicAssumption
from idaes.core.util.model_statistics import degrees_of_freedom
import idaes.logger as idaeslog
import hda_ideal_VLE as thermo_props
import hda_reaction as reaction_props

In [2]:
m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)

In [3]:
m.fs.thermo_params = thermo_props.HDAParameterBlock()
m.fs.reaction_params = reaction_props.HDAReactionParameterBlock(
                        property_package=m.fs.thermo_params)

'fs.thermo_params.pressure_crit' to mutable.
'fs.thermo_params.temperature_crit' to mutable.
'fs.thermo_params.mw_comp' to mutable.
'fs.thermo_params.dens_liq_param_1' to mutable.
'fs.thermo_params.dens_liq_param_2' to mutable.
'fs.thermo_params.dens_liq_param_3' to mutable.
'fs.thermo_params.dens_liq_param_4' to mutable.
'fs.thermo_params.temperature_boil' to mutable.
'fs.thermo_params.cp_ig_1' to mutable.
'fs.thermo_params.cp_ig_2' to mutable.
'fs.thermo_params.cp_ig_3' to mutable.
'fs.thermo_params.cp_ig_4' to mutable.
'fs.thermo_params.cp_ig_5' to mutable.
'fs.thermo_params.pressure_sat_coeff_A' to mutable.
'fs.thermo_params.pressure_sat_coeff_B' to mutable.
'fs.thermo_params.pressure_sat_coeff_C' to mutable.
'fs.thermo_params.dh_vap' to mutable.


In [4]:
m.fs.M101 = Mixer(property_package=m.fs.thermo_params,
                  inlet_list=["toluene_feed", "hydrogen_feed", "vapor_recycle"])

m.fs.H101 = Heater(property_package=m.fs.thermo_params,
                   has_pressure_change=False,
                   has_phase_equilibrium=True)
m.fs.R101 = StoichiometricReactor(
            property_package=m.fs.thermo_params,
            reaction_package=m.fs.reaction_params,
            has_heat_of_reaction=True,
            has_heat_transfer=True,
            has_pressure_change=False)
m.fs.F101 = Flash(property_package=m.fs.thermo_params,
                  has_heat_transfer=True,
                  has_pressure_change=True)
m.fs.S101 = Splitter(property_package=m.fs.thermo_params,
                     ideal_separation=False,
                     outlet_list=["purge", "recycle"])
m.fs.C101 = PressureChanger(
            property_package=m.fs.thermo_params,
            compressor=True,
            thermodynamic_assumption=ThermodynamicAssumption.isothermal)
    
m.fs.F102 = Flash(property_package=m.fs.thermo_params,
                  has_heat_transfer=True,
                  has_pressure_change=True)

In [5]:
m.fs.s01 = Arc(source=m.fs.M101.toluene_feed, destination=m.fs.M101.toluene_feed)
m.fs.s02 = Arc(source=m.fs.M101.hydrogen_feed, destination=m.fs.M101.hydrogen_feed)
m.fs.s03 = Arc(source=m.fs.M101.outlet, destination=m.fs.H101.inlet)
m.fs.s04 = Arc(source=m.fs.H101.outlet, destination=m.fs.R101.inlet)
m.fs.s05 = Arc(source=m.fs.R101.outlet, destination=m.fs.F101.inlet)
m.fs.s06 = Arc(source=m.fs.F101.vap_outlet, destination=m.fs.S101.inlet)
m.fs.s08 = Arc(source=m.fs.S101.recycle, destination=m.fs.C101.inlet)
m.fs.s09 = Arc(source=m.fs.C101.outlet, destination=m.fs.M101.vapor_recycle)
m.fs.s10 = Arc(source=m.fs.F101.liq_outlet, destination=m.fs.F102.inlet)

In [6]:
TransformationFactory("network.expand_arcs").apply_to(m)

In [7]:
print(degrees_of_freedom(m))

9


In [8]:
t_feed = m.fs.M101.toluene_feed
t_feed.flow_mol_phase_comp[0, "Vap", "benzene"].fix(1e-5)
t_feed.flow_mol_phase_comp[0, "Vap", "toluene"].fix(1e-5)
t_feed.flow_mol_phase_comp[0, "Vap", "hydrogen"].fix(1e-5)
t_feed.flow_mol_phase_comp[0, "Vap", "methane"].fix(1e-5)
t_feed.flow_mol_phase_comp[0, "Liq", "benzene"].fix(1e-5)
t_feed.flow_mol_phase_comp[0, "Liq", "toluene"].fix(0.30)
t_feed.flow_mol_phase_comp[0, "Liq", "hydrogen"].fix(1e-5)
t_feed.flow_mol_phase_comp[0, "Liq", "methane"].fix(1e-5)
t_feed.temperature.fix(303.2)
t_feed.pressure.fix(350000)

In [9]:
h_feed = m.fs.M101.hydrogen_feed
h_feed.flow_mol_phase_comp[0, "Vap", "benzene"].fix(1e-5)
h_feed.flow_mol_phase_comp[0, "Vap", "toluene"].fix(1e-5)
h_feed.flow_mol_phase_comp[0, "Vap", "hydrogen"].fix(0.30)
h_feed.flow_mol_phase_comp[0, "Vap", "methane"].fix(0.02)
h_feed.flow_mol_phase_comp[0, "Liq", "benzene"].fix(1e-5)
h_feed.flow_mol_phase_comp[0, "Liq", "toluene"].fix(1e-5)
h_feed.flow_mol_phase_comp[0, "Liq", "hydrogen"].fix(1e-5)
h_feed.flow_mol_phase_comp[0, "Liq", "methane"].fix(1e-5)
h_feed.temperature.fix(303.2)
h_feed.pressure.fix(350000)

In [10]:
m.fs.H101.outlet.temperature.fix(600)

In [11]:
m.fs.R101.conversion = Var(initialize=0.75, bounds=(0, 1))

m.fs.R101.conv_constraint = Constraint(
    expr=m.fs.R101.conversion*m.fs.R101.inlet.
    flow_mol_phase_comp[0, "Vap", "toluene"] ==
    (m.fs.R101.inlet.flow_mol_phase_comp[0, "Vap", "toluene"] -
     m.fs.R101.outlet.flow_mol_phase_comp[0, "Vap", "toluene"]))

m.fs.R101.conversion.fix(0.75)
m.fs.R101.heat_duty.fix(0)

In [12]:
m.fs.F101.vap_outlet.temperature.fix(325.0)
m.fs.F101.deltaP.fix(0)

In [13]:
m.fs.F102.vap_outlet.temperature.fix(375)
m.fs.F102.deltaP.fix(-200000)

In [14]:
m.fs.S101.split_fraction[0, "purge"].fix(0.2)
m.fs.C101.outlet.pressure.fix(350000)

In [17]:
print(degrees_of_freedom(m))

-20


In [ ]:
m.fs.report()